# Random Forests & Extra Trees: Ensembling, Feature Importance, and Permutation Testing
Accompanies the notes on **Bagging, Random Forests and Extra Trees**.

The variance of an ensemble of $B$ trees with individual variance $\sigma^2$ and pairwise correlation $\rho$ is:
$$\text{Var}(\bar{f}) = \rho \sigma^2 + \frac{1 - \rho}{B} \sigma^2$$

Random Forests drive ensemble variance down through two orthogonal mechanisms:
1. **Bootstrap Aggregation (Bagging)**: Each tree trains on an independent bootstrap sample with replacement.
2. **Feature Subsampling ($m = \sqrt{p}$)**: At each split, only a random subset of features is evaluated, decorrelating the trees (reducing $\rho$).

In this notebook, we tune Random Forest hyperparameters and compare **Mean Decrease in Impurity (MDI)** against **Permutation Feature Importance** on holdout test data.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.inspection import permutation_importance
from scipy.stats import randint

data_path = os.path.abspath('../../data/diabetic_data.csv')
df = pd.read_csv(data_path)

numeric_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 
                'num_medications', 'number_outpatient', 'number_emergency', 
                'number_inpatient', 'number_diagnoses']
X = df[numeric_cols].fillna(0)
y = (df['readmitted'] == '<30').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Hyperparameter Tuning Rationale:
- `n_estimators`: Adding more trees *never* causes overfitting in Random Forests. It monotonically reduces variance until reaching an asymptotic plateau (typically $100-300$ trees).
- `max_features`: Evaluating fewer features per split decorrelates trees (lowers $\rho$), but setting it too small degrades individual tree strength. Standard default is $\sqrt{p}$.
- `min_samples_leaf`: Enforces smoothing on leaf predictions.

In [ ]:
param_dist = {
    'n_estimators': randint(80, 220),
    'max_depth': randint(5, 16),
    'max_features': ['sqrt', 'log2', 0.5],
    'min_samples_leaf': randint(2, 10)
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, oob_score=True, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=15,
    cv=3,
    random_state=42,
    n_jobs=-1
)
rf_search.fit(X_train, y_train)

print(f"Best CV ROC-AUC: {rf_search.best_score_:.4f}")
print(f"Optimal Parameters: {rf_search.best_params_}")
best_rf = rf_search.best_estimator_
print(f"Out-Of-Bag (OOB) Accuracy: {best_rf.oob_score_:.4f}")

### Comparing Feature Importances: Impurity (MDI) vs. Permutation
- **MDI (Impurity)**: Measures total Gini reduction in training trees. Vulnerable to continuous feature bias.
- **Permutation Importance**: Shuffles column $j$ on **unseen test data** and measures the drop in accuracy. Model-agnostic and immune to in-sample overfitting!

In [ ]:
# 1. MDI Feature Importance
mdi_importances = best_rf.feature_importances_

# 2. Permutation Importance on holdout test set
perm_result = permutation_importance(best_rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
perm_importances = perm_result.importances_mean

# Comparison plot
plt.figure(figsize=(11, 4.5))
x_indices = np.arange(len(numeric_cols))
bar_width = 0.35

plt.bar(x_indices - bar_width/2, mdi_importances, width=bar_width, label='MDI (In-Sample Impurity)', color='royalblue')
plt.bar(x_indices + bar_width/2, perm_importances, width=bar_width, label='Permutation (Test Holdout Drop)', color='crimson')

plt.xticks(x_indices, numeric_cols, rotation=35, ha='right', fontsize=9)
plt.ylabel('Importance Score', fontsize=11)
plt.title('MDI vs. Permutation Feature Importance Comparison', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## The bank data: one tree vs bagging vs random forest vs Extra Trees (from the notes)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

bd = pd.read_csv("../../data/bd_train.csv").drop(columns=["REF_NO", "post_code", "post_area"])
y = (bd.pop("Revenue.Grid") == 1).astype(int)
X = pd.get_dummies(bd, drop_first=True, dtype=float)
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)
print(x_train.shape)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, ExtraTreesClassifier

models = {
    "single deep tree": DecisionTreeClassifier(random_state=0),
    "bagged trees (all features)": BaggingClassifier(DecisionTreeClassifier(), n_estimators=300,
                                                     oob_score=True, random_state=0, n_jobs=-1),
    "random forest (sqrt features)": RandomForestClassifier(n_estimators=300, max_features="sqrt",
                                                            oob_score=True, random_state=0, n_jobs=-1),
    "extra trees": ExtraTreesClassifier(n_estimators=300, max_features="sqrt", random_state=0, n_jobs=-1),
}
for name, m in models.items():
    m.fit(x_train, y_train)
    oob = f"  OOB accuracy {m.oob_score_:.3f}" if hasattr(m, "oob_score_") else ""
    print(f"{name:30s} test AUC {roc_auc_score(y_test, m.predict_proba(x_test)[:, 1]):.4f}{oob}")

## Measuring the tree correlation rho

In [ ]:
def tree_correlation(ensemble, n_pairs=200, seed=0):
    preds = np.array([t.predict_proba(x_test.values)[:, 1] for t in ensemble.estimators_])   # one row per tree
    rng = np.random.default_rng(seed)
    pairs = rng.choice(len(preds), size=(n_pairs, 2))
    return np.mean([np.corrcoef(preds[i], preds[j])[0, 1] for i, j in pairs if i != j])

print("average correlation between two trees' predictions:")
print("  bagging      ", round(tree_correlation(models["bagged trees (all features)"]), 3))
print("  random forest", round(tree_correlation(models["random forest (sqrt features)"]), 3))
print("  extra trees  ", round(tree_correlation(models["extra trees"]), 3))

## How many trees?

In [ ]:
for n in [1, 5, 25, 100, 300]:
    rf = RandomForestClassifier(n_estimators=n, max_features="sqrt", random_state=0, n_jobs=-1).fit(x_train, y_train)
    print(f"{n:4d} trees: test AUC {roc_auc_score(y_test, rf.predict_proba(x_test)[:, 1]):.4f}")

## How many features per split?

In [ ]:
for mf in [1, 3, "sqrt", 0.3, 1.0]:
    rf = RandomForestClassifier(n_estimators=300, max_features=mf, oob_score=True, random_state=0, n_jobs=-1).fit(x_train, y_train)
    print(f"max_features={str(mf):5s}: OOB accuracy {rf.oob_score_:.4f}  test AUC {roc_auc_score(y_test, rf.predict_proba(x_test)[:, 1]):.4f}")

## The out-of-bag fraction

In [ ]:
for N in [2, 5, 10, 100, 1000, 6093]:
    print(N, round((1 - 1 / N) ** N, 4))
print('1/e =', round(np.exp(-1), 6))

## Try it yourself
1. Variance formula with rho = 0.651 and 0.325, B = 10 and 300.
2. ExtraTreesClassifier with bootstrap=True, oob_score=True.
3. feature_importances_ vs permutation importance on the test set: do the top five agree?